# Makemore Part 2: Multi-Layer Perceptron (MLP)

This notebook builds a character-level language model that predicts the next character using a multi-layer perceptron.

## Learning objectives
- Turn a raw list of names into (context, target) training pairs.
- Build the forward pass of an MLP from scratch: embedding lookup, hidden layer, softmax, and cross-entropy loss.
- Train the model with mini-batch stochastic gradient descent and measure generalization on train/validation/test splits.
- Visualize learned character embeddings and sample new names from the trained model.

## Why this matters
MLPs are the simplest "deep" neural network. Seeing how embedding, matrix multiplication, non-linearity, and loss fit together prepares you for modern architectures (RNNs, Transformers, and beyond) while keeping the math transparent.

## Prerequisites
- Python and NumPy basics
- PyTorch tensors and automatic differentiation
- The bigram model from Part 1 (optional but helpful)

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from nnzero import build_vocab, build_dataset, split_dataset, load_names, set_seed
%matplotlib inline

## Quick-run mode for CI

When the environment variable `NNZERO_QUICK_RUN=1` is set, training loops run for only a small number of steps so that notebooks can be validated quickly. Students should leave this unset to train the full models.

In [ ]:
import os
QUICK_RUN = os.environ.get("NNZERO_QUICK_RUN", "0") == "1"
if QUICK_RUN:
    print("Quick-run mode enabled: training loops will use far fewer steps.")

> **Note:** `build_vocab`, `build_dataset`, `split_dataset`, `load_names`, and `set_seed` are implemented from scratch in the lecture videos. We import them from the local `nnzero` package to keep this notebook focused on the MLP, but the source code is in `nnzero/utils.py` if you want to see how they work.

In [ ]:
# Load the names dataset from the repo root
words = load_names('data/names.txt')
words[:8]

In [ ]:
len(words)

## From words to numbers
A neural network cannot read strings directly. We first build a vocabulary that maps every character to a unique integer, then create training examples where a context of characters predicts the next one.

In [ ]:
# Build character-to-index and index-to-character mappings
stoi, itos, vocab_size = build_vocab(words)
print(f"vocab size: {vocab_size}")
print(itos)

## Why split into train / validation / test?
We train on the training set, tune decisions (like model size or learning rate) on the validation set, and report final performance on the test set. This tells us whether the model actually generalizes or just memorizes the training names.

In [ ]:
# Split names and build (context, target) tensors
block_size = 3  # context length: how many characters do we take to predict the next one?

train_words, val_words, test_words = split_dataset(words)
Xtr, Ytr = build_dataset(train_words, stoi, block_size=block_size)
Xdev, Ydev = build_dataset(val_words, stoi, block_size=block_size)
Xte, Yte = build_dataset(test_words, stoi, block_size=block_size)

In [ ]:
Xtr.shape, Ytr.shape # dataset

## Building the forward pass from scratch
Before calling `torch.nn`, it is worth seeing every operation explicitly: lookup the embedding vectors, flatten the context, multiply by weights, apply a non-linearity, and produce logits. This is the core of the MLP.

In [ ]:
C = torch.randn((27, 2))

In [ ]:
emb = C[Xtr]
emb.shape


In [ ]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [ ]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [ ]:
h

In [ ]:
h.shape

In [ ]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [ ]:
logits = h @ W2 + b2

In [ ]:
logits.shape

## Why softmax + negative log-likelihood?
The softmax turns logits into a probability distribution over characters. We then maximize the log-probability of the correct next character, which is the same as minimizing the cross-entropy loss. `F.cross_entropy` computes this in one numerically stable step.

In [ ]:
counts = logits.exp()

In [ ]:
prob = counts / counts.sum(1, keepdims=True)

In [ ]:
prob.shape

In [ ]:
loss = -prob[torch.arange(32), Ytr[:32]].log().mean()
loss


## From a one-off forward pass to a real training loop

## Why small random weights?
Random weights break symmetry so different hidden units learn different things. Keeping them small (and, in deeper networks, scaling by `1 / sqrt(fan_in)` with Kaiming init) prevents activations from exploding or vanishing at the start of training.

In [ ]:
# Initialize parameters with a reproducible generator
g = set_seed(2147483647)
C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 200), generator=g)
b1 = torch.randn(200, generator=g)
W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [ ]:
sum(p.nelement() for p in parameters) # number of parameters in total

In [ ]:
for p in parameters:
  p.requires_grad = True

## Why mini-batch SGD?
Computing the loss on the whole dataset every update is expensive. A mini-batch gives a noisy but cheap estimate of the gradient, and taking many small steps usually converges faster than a few exact steps. We decay the learning rate later so the model can settle into a minimum.

PyTorch's autograd builds the computation graph and traverses it in reverse topological order during `backward()`, so every parameter receives the correct gradient via the chain rule.

In [ ]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [ ]:
lri = []
lossi = []
stepi = []

In [ ]:
for i in range(200000):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (32,))
  
  # forward pass
  emb = C[Xtr[ix]] # (32, 3, 10)
  h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 200)
  logits = h @ W2 + b2 # (32, 27)
  loss = F.cross_entropy(logits, Ytr[ix])
  #print(loss.item())
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  #lr = lrs[i]
  lr = 0.1 if i < 100000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  #lri.append(lre[i])
  stepi.append(i)
  lossi.append(loss.log10().item())

#print(loss.item())

In [ ]:
plt.plot(stepi, lossi)

In [ ]:
emb = C[Xtr] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ytr)
loss

In [ ]:
emb = C[Xdev] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ydev)
loss

## Why visualize embeddings?
After training, similar characters (vowels, consonants, punctuation) often end up close together in embedding space. The plot gives intuition about what the model has learned.

In [ ]:
# visualize dimensions 0 and 1 of the embedding matrix C for all characters
plt.figure(figsize=(8,8))
plt.scatter(C[:,0].data, C[:,1].data, s=200)
for i in range(C.shape[0]):
    plt.text(C[i,0].item(), C[i,1].item(), itos[i], ha="center", va="center", color='white')
plt.grid('minor')

## 🏋️ Try it yourself: change the model size

The notebook uses an embedding dimension of `10` and a hidden layer of `200` neurons. Try a different configuration (for example, embedding dimension `20` and hidden size `100`) and re-train. Compare the final train and dev loss. Does a bigger model always generalize better?

In [ ]:
# Your code here


## 💡 Solution

In [ ]:
# Example: larger embedding, smaller hidden layer
# (Assumes block_size = 3, so the flattened input size is 3 * 20 = 60.)
g = set_seed(2147483647)
C = torch.randn((27, 20), generator=g)
W1 = torch.randn((60, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad = True

stepi, lossi = [], []
for i in range(200000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    emb = C[Xtr[ix]]                       # (32, 3, 20)
    h = torch.tanh(emb.view(-1, 60) @ W1 + b1)  # (32, 100)
    logits = h @ W2 + b2                   # (32, 27)
    loss = F.cross_entropy(logits, Ytr[ix])

    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    stepi.append(i)
    lossi.append(loss.log10().item())

emb_tr = C[Xtr]
h_tr = torch.tanh(emb_tr.view(-1, 60) @ W1 + b1)
logits_tr = h_tr @ W2 + b2
print("train loss:", F.cross_entropy(logits_tr, Ytr).item())

emb_dev = C[Xdev]
h_dev = torch.tanh(emb_dev.view(-1, 60) @ W1 + b1)
logits_dev = h_dev @ W2 + b2
print("dev loss:", F.cross_entropy(logits_dev, Ydev).item())

In [ ]:
context = [0] * block_size
C[torch.tensor([context])].shape

## Why sample instead of taking the argmax?
Language generation is probabilistic: at each step we draw a character from the predicted distribution. Sampling produces diverse, name-like strings; always picking the most likely character would give repetitive output.

In [ ]:


# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

## 🏋️ Try it yourself: temperature sampling

The sampling loop uses `F.softmax(logits, dim=1)` with temperature `1.0`. Add a `temperature` variable and divide the logits by it before the softmax. Generate 20 names each for temperatures `0.5`, `1.0`, and `2.0`. What changes?

In [ ]:
# Your code here


## 💡 Solution

In [ ]:
def sample_names(temperature: float, n: int = 20) -> list[str]:
    g = torch.Generator().manual_seed(2147483647 + 10)
    results = []
    for _ in range(n):
        out = []
        context = [0] * block_size
        while True:
            emb = C[torch.tensor([context])]        # (1, block_size, d)
            h = torch.tanh(emb.view(1, -1) @ W1 + b1)
            logits = h @ W2 + b2
            probs = F.softmax(logits / temperature, dim=1)
            ix = torch.multinomial(probs, num_samples=1, generator=g).item()
            context = context[1:] + [ix]
            out.append(ix)
            if ix == 0:
                break
        results.append(''.join(itos[i] for i in out))
    return results

for t in [0.5, 1.0, 2.0]:
    print(f"--- temperature {t} ---")
    for name in sample_names(t, 5):
        print(name)